# 00. Generate low-fidelity synthetic SAMPLE.csv

This notebook creates `data/SAMPLE.csv`, a synthetic monthly multisector default-count panel.

The sample is not derived from proprietary empirical data and is intended only to make the reproduction workflow executable.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit

DATA = Path("data")
DATA.mkdir(exist_ok=True)
RNG = np.random.default_rng(20260618)

In [ ]:
sectors = [
    "Basic Materials",
    "Capital Goods",
    "Consumer",
    "Energy",
    "Finance",
    "Healthcare",
    "Technology",
    "Utilities",
]
dates = pd.date_range("2000-01-31", periods=312, freq=pd.offsets.MonthEnd())
T = len(dates)
S = len(sectors)

phis = np.array([0.88, 0.62])
innov = np.array([0.34, 0.24])
factors = np.zeros((T, 2))
for t in range(1, T):
    factors[t] = phis * factors[t - 1] + RNG.normal(0.0, innov)

loadings = np.array([
    [0.45, 0.28],
    [0.42, 0.10],
    [0.30, -0.20],
    [0.52, 0.42],
    [0.62, -0.12],
    [0.22, -0.32],
    [0.36, -0.38],
    [0.18, 0.20],
])
loadings = loadings / np.linalg.norm(loadings, axis=0, keepdims=True)

baseline = np.array([-5.75, -5.55, -5.85, -5.35, -5.15, -6.05, -5.90, -6.15])
seasonal = 0.10 * np.sin(2 * np.pi * np.arange(T) / 12.0)
sigma_eps_common = 0.26

rows = []
for t, date in enumerate(dates):
    cycle = factors[t] @ loadings.T
    eps = RNG.normal(0.0, sigma_eps_common, size=S)
    eta = baseline + cycle + seasonal[t] + eps
    p = expit(eta)
    for s, sector in enumerate(sectors):
        trend = 1.0 + 0.08 * np.sin(2 * np.pi * t / 96.0 + s / 3.0)
        obligors = int(np.round(RNG.normal(1600 + 80 * s, 70) * trend))
        obligors = max(obligors, 600)
        defaulted = int(RNG.binomial(obligors, p[s]))
        rows.append({
            "date": date.strftime("%Y-%m-%d"),
            "sector": sector,
            "obligors": obligors,
            "defaulted": defaulted,
        })

sample = pd.DataFrame(rows)
sample.to_csv(DATA / "SAMPLE.csv", index=False)
sample.head()

In [ ]:
summary = sample.assign(rate=sample["defaulted"] / sample["obligors"]).groupby("sector").agg(
    months=("date", "nunique"),
    obligors=("obligors", "sum"),
    defaulted=("defaulted", "sum"),
    mean_monthly_rate=("rate", "mean"),
)
summary